# Solução do Trabalho Prático: TF-IDF e Similaridade de Cosseno

Neste notebook, implementaremos o cálculo de TF-IDF e da similaridade de cosseno de duas formas:
1. Manualmente usando Pandas e NumPy.
2. Usando o `scikit-learn` para validar nossos resultados.

In [3]:
import os
import glob
import unicodedata
import string
import math
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Leitura dos Documentos e Pré-processamento

Os relatórios estão localizados no diretório `corpora/`. Vamos ler e realizar o pré-processamento (normalização unicode, tokenização e remoção de stop words).

In [6]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('portuguese'))
corpora_dir = '/home/wagnerperin/fucape/disciplinas/pln-sistemas-ri/01_planos_de_aulas/repositorio_aula02/corpora'

def preprocess(text):
    # Converter para minúsculo
    text = text.lower()
    # Normalização unicode (remover acentos)
    text = ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')
    # Remover pontuação
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenização e filtragem de stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

# Leitura dos arquivos
documents = {}
file_paths = glob.glob(os.path.join(corpora_dir, '*.txt'))
for path in sorted(file_paths):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
        filename = os.path.basename(path)
        documents[filename] = content

# Se o diretório estiver vazio (para demonstração, vamos criar alguns docs dummy)
if not documents:
    print("Nenhum arquivo encontrado. Criando documentos de exemplo...")
    for i in range(1, 11):
        documents[f'relatorio_{i}.txt'] = f"Este é o contrato de prestação de serviços {i} com rescisão e multa."

doc_ids = list(documents.keys())
corpus = [documents[doc_id] for doc_id in doc_ids]
tokenized_corpus = [preprocess(doc) for doc in corpus]

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/wagnerperin/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 2. Construção Manual: TF, IDF e TF-IDF

Cálculo de:
- $TF(t,d) = 1 + \log_{10}(f_{t,d})$
- $IDF(t) = \log_{10}(N / DF_t)$

In [7]:
# Vocabulário único
vocab = set()
for tokens in tokenized_corpus:
    vocab.update(tokens)
vocab = sorted(list(vocab))

# Contagem de frequências (TF cru)
tf_raw = {doc_id: Counter(tokens) for doc_id, tokens in zip(doc_ids, tokenized_corpus)}

N = len(doc_ids)

# Calcular IDF
df = {}
for term in vocab:
    df[term] = sum(1 for tokens in tokenized_corpus if term in tokens)

idf = {term: math.log10(N / df[term]) for term in vocab}

# Matriz TF-IDF Manual
tfidf_manual = pd.DataFrame(index=doc_ids, columns=vocab).fillna(0.0)

for doc_id in doc_ids:
    for term in vocab:
        f_td = tf_raw[doc_id].get(term, 0)
        if f_td > 0:
            tf = 1 + math.log10(f_td)
            tfidf_manual.loc[doc_id, term] = tf * idf[term]

display(tfidf_manual.head())

,antecipada,aplicacao,auditoria,causa,clausula,cobranca,confidencialidade,contrato,contratual,corporativo,...,sigilo,sobre,software,suporte,suspensao,telecomunicacao,termo,ti,trabalho,violacao
relatorio_01.txt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.69897,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
relatorio_02.txt,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.69897,0.522879,0.0,0.0,0.0,0.0,0.0,0.0
relatorio_03.txt,1.0,0.0,0.0,0.0,0.69897,0.0,0.0,0.0,0.69897,0.0,...,0.0,0.69897,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
relatorio_04.txt,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
relatorio_05.txt,0.0,0.0,0.0,0.0,0.69897,0.0,0.0,0.522879,0.0,0.0,...,0.0,0.0,0.0,0.522879,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Similaridade de Cosseno Manual

Consultas:
- `q1 = "contrato de prestação de serviços com cláusula de suporte"`
- `q2 = "rescisão antecipada de contrato e multa"`

In [8]:
def vectorize_query_manual(query, vocab, idf):
    tokens = preprocess(query)
    tf_raw_q = Counter(tokens)
    q_vector = np.zeros(len(vocab))
    for i, term in enumerate(vocab):
        f_tq = tf_raw_q.get(term, 0)
        if f_tq > 0:
            tf = 1 + math.log10(f_tq)
            q_vector[i] = tf * idf.get(term, 0)
    return q_vector

def cosine_sim(vec_a, vec_b):
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(vec_a, vec_b) / (norm_a * norm_b)

queries = [
    "contrato de prestação de serviços com cláusula de suporte",
    "rescisão antecipada de contrato e multa"
]

doc_matrix = tfidf_manual.values
results_manual = []

for q in queries:
    q_vec = vectorize_query_manual(q, vocab, idf)
    sims = [cosine_sim(q_vec, doc_vec) for doc_vec in doc_matrix]
    results_manual.append(sims)
    
    print(f"Consulta: {q}")
    for doc_id, sim in zip(doc_ids, sims):
        if sim > 0:
            print(f"  {doc_id}: {sim:.4f}")
    print()

Consulta: contrato de prestação de serviços com cláusula de suporte
  relatorio_01.txt: 0.2599
  relatorio_02.txt: 0.1054
  relatorio_03.txt: 0.2015
  relatorio_05.txt: 1.0000
  relatorio_06.txt: 0.1082
  relatorio_10.txt: 0.4943

Consulta: rescisão antecipada de contrato e multa
  relatorio_01.txt: 0.1146
  relatorio_03.txt: 0.6407
  relatorio_05.txt: 0.1524
  relatorio_06.txt: 0.1908
  relatorio_08.txt: 0.1465
  relatorio_10.txt: 0.1090



## 4. Validação com Scikit-Learn

Usando `TfidfVectorizer` e comparando. O scikit-learn utiliza implementações ligeiramente diferentes por padrão (ex. base neperiana para o logaritmo no IDF e vetorização com normalização L2 nos documentos). Aqui tentamos simular ao máximo as configurações usadas.

In [9]:
# Preprocessar corpus para o sklearn como strings unidas
corpus_preprocessed = [' '.join(preprocess(doc)) for doc in corpus]
queries_preprocessed = [' '.join(preprocess(q)) for q in queries]

# sublinear_tf=True aplica 1 + log(tf)
# norm=None para usar as pontuações sem normalizar por L2, e depois faremos normalização cosseno.
vectorizer = TfidfVectorizer(sublinear_tf=True, norm=None, smooth_idf=False)
X = vectorizer.fit_transform(corpus_preprocessed)

# Calcular cosseno comparado com as consultas
X_queries = vectorizer.transform(queries_preprocessed)
sim_sklearn = cosine_similarity(X_queries, X)

for i, q in enumerate(queries):
    print(f"Consulta (Sklearn): {q}")
    for j, doc_id in enumerate(doc_ids):
        sim = sim_sklearn[i][j]
        if sim > 0:
            print(f"  {doc_id}: {sim:.4f}")
    print()

Consulta (Sklearn): contrato de prestação de serviços com cláusula de suporte
  relatorio_01.txt: 0.3033
  relatorio_02.txt: 0.1325
  relatorio_03.txt: 0.1883
  relatorio_05.txt: 1.0000
  relatorio_06.txt: 0.1351
  relatorio_10.txt: 0.5906

Consulta (Sklearn): rescisão antecipada de contrato e multa
  relatorio_01.txt: 0.1483
  relatorio_03.txt: 0.6107
  relatorio_05.txt: 0.1811
  relatorio_06.txt: 0.2643
  relatorio_08.txt: 0.1624
  relatorio_10.txt: 0.1444

